# Walkthrough: one example, end to end, on both sides

This notebook takes a single small example through the two ways of deciding whether an RDF graph `G` entails a graph `H` that the paper *Implication-Space Semantics for RDF* proves equivalent (Theorem 2):

* **the practice's side** (Definition 4): apply the regime's rules to `G` until nothing new appears, giving the closure cl_R(G); then look for a copy of `H` in it (simple entailment, Definition 3), or for ⊥;
* **the semantics' side** (Definitions 6–14): build the content of `G` and the content of `H`, adjoin `G`'s positive role to `H`'s negative role, and ask whether every generating pair of the result is a good implication of the frame 𝕀_R that the regime induces.

Every object on the way is printed. Nothing is computed in this notebook itself: all functions come from the `issrdf` package, one module per group of definitions (see `DEFINITIONS.md`). The notebook uses the paper's own example (tweety, Bird, Flier; its question 2) and the two rules it names, rdfs9 and cax-dw.

Four cases: **A** ground `G` and `H`; **B** a blank node in `H`; **C** a blank node in `G`, where the Skolem instance decides; **D** an incoherent `G` (Proposition 2).

In [1]:
import sys; sys.path.insert(0, '..')
from issrdf import (Universe, BOT, make_closure, simply_entails, r_entails, r_inconsistent,
                    ground_content, mappings, instances, content_pos, content_neg,
                    adj, adj_iter, UNIT, make_good, iss_entails, iss_incoherent, show)
from issrdf.show import triple, graph, pair, pairs, regime, mapping, report, why, union_labels, verdict_line

## The universe and the regime

**Convention 1** fixes a finite fragment `N` of names over which the frame is built. Here `N` has seven names: the three individuals of the example, the three IRIs of the regime's vocabulary `V`, and one spare IRI `s`. The spare is required by **Definition 12** (admissibility): there must be an injection from the blank nodes of `G` into IRIs of `N` that occur neither in `G`, nor in `H`, nor in `V`. With one blank node in `G`, one spare suffices.

The regime `R` has two rule schemas (Remark 5: a schema stands for all its instances under substitution for the metavariables `X`, `Y`, `Z`; with no side conditions, uniformity is automatic):

* **rdfs9**: `{(X, type, Y), (Y, subClassOf, Z)} → (X, type, Z)`
* **cax-dw** (OWL 2 RL, Remark 6): `{(X, type, Y), (X, type, Z), (Y, disjointWith, Z)} → ⊥`

`V` is the set of IRIs the rules mention: `type`, `subClassOf`, `disjointWith`.

In [2]:
U = Universe(V=('type', 'subClassOf', 'disjointWith'),
             INDIV=('tweety', 'Bird', 'Flier'),
             SPARE=('s',),
             BN_G=('_x',), BN_H=('_y',))
R = (
    ((('X', 'type', 'Y'), ('Y', 'subClassOf', 'Z')), ('X', 'type', 'Z')),                      # rdfs9
    ((('X', 'type', 'Y'), ('X', 'type', 'Z'), ('Y', 'disjointWith', 'Z')), BOT),                # cax-dw
)
closure = make_closure(R, U.V)       # cl_R, Definition 4
good = make_good(closure)            # membership in I_R, Definitions 8, 13, 14 (Herbrand, Def. 15)
print(U); print(); print(regime(R))
verdicts = []                        # collected for the summary and the log

Universe(V=('type', 'subClassOf', 'disjointWith'), INDIV=('tweety', 'Bird', 'Flier'), SPARE=('s',), BN_G=('_x',), BN_H=('_y',))  # N = ('tweety', 'Bird', 'Flier', 'type', 'subClassOf', 'disjointWith', 's')

{(X, type, Y), (Y, subClassOf, Z)} → (X, type, Z)
{(X, type, Y), (X, type, Z), (Y, disjointWith, Z)} → ⊥


## Case A: ground `G`, ground `H`

`G` = {(tweety, type, Bird), (Bird, subClassOf, Flier)}, `H` = {(tweety, type, Flier)}.

**Practice's side.** cl_R(G) adds (tweety, type, Flier) by rdfs9. `H` is a subgraph of it, so `G` R-entails `H` (Definition 4; for ground `H`, simple entailment is inclusion).

**Semantics' side.** By **Definition 10** the content of a ground graph is ⟦G⟧ = ⟨⊔{𝓡⁺(t)}, ∇{𝓡⁻(t)}⟩. On generating sets (Definition 6) the positive role is generated by the single pair ⟨G, ∅⟩, and the negative role by the pairs ⟨∅, S⟩ for every non-empty S ⊆ G: that is what the power-symjunction ∇ does, and it is what makes a graph in succedent position the denial of its triples severally and jointly. Content entailment (**Definition 7**, via Lemma 2) asks whether every generating pair of ⟦G⟧⁺ ⊔ ⟦H⟧⁻ lies in 𝕀_R. Here there is exactly one such pair, ⟨G, {(tweety, type, Flier)}⟩, and it is in 𝕀_R because (tweety, type, Flier) ∈ cl_R(G) (**Definition 13**).

In [3]:
G_a = frozenset({('tweety', 'type', 'Bird'), ('Bird', 'subClassOf', 'Flier')})
H_a = frozenset({('tweety', 'type', 'Flier')})
assert U.admissible(G_a, H_a)

print("G =", graph(G_a)); print("H =", graph(H_a))
print("\n-- practice's side (Definition 4) --")
cl = closure(G_a)
print("cl_R(G) =", graph(cl))
print("cl_R(G) \\ G =", graph(cl - G_a))
print("cl_R(G) simply entails H:", simply_entails(cl - {BOT}, H_a))
rules_a = r_entails(closure, G_a, H_a)

print("\n-- semantics' side (Definitions 10, 7, 13) --")
posG, negG = ground_content(G_a)
print("[[G]]+ generated by:"); print(pairs(posG))
print("[[G]]- generated by:"); print(pairs(negG))
posH, negH = ground_content(H_a)
print("[[H]]- generated by:"); print(pairs(negH))
F = adj(posG, negH)
print(f"[[G]]+ ⊔ [[H]]-: {len(F)} pair(s); test each against I_R:")
sem_a = report(closure, F)

print(); print(verdict_line("A: ground G, ground H", sem_a, rules_a))
verdicts.append(("A: ground G, ground H", sem_a, rules_a))

G = {(Bird, subClassOf, Flier), (tweety, type, Bird)}
H = {(tweety, type, Flier)}

-- practice's side (Definition 4) --
cl_R(G) = {(Bird, subClassOf, Flier), (tweety, type, Bird), (tweety, type, Flier)}
cl_R(G) \ G = {(tweety, type, Flier)}
cl_R(G) simply entails H: True

-- semantics' side (Definitions 10, 7, 13) --
[[G]]+ generated by:
⟨{(Bird, subClassOf, Flier), (tweety, type, Bird)}, ∅⟩
[[G]]- generated by:
⟨∅, {(Bird, subClassOf, Flier)}⟩
⟨∅, {(tweety, type, Bird)}⟩
⟨∅, {(Bird, subClassOf, Flier), (tweety, type, Bird)}⟩
[[H]]- generated by:
⟨∅, {(tweety, type, Flier)}⟩
[[G]]+ ⊔ [[H]]-: 1 pair(s); test each against I_R:
  ✓ ⟨{(Bird, subClassOf, Flier), (tweety, type, Bird)}, {(tweety, type, Flier)}⟩
      (tweety, type, Flier) ∈ Δ ∩ cl_R(Γ)  (Definition 13)

A: ground G, ground H                    semantics True   rules True   agree


## Case B: a blank node in `H`

`H` = {(_:y, type, Flier)}: "something is a Flier".

**Practice's side.** cl_R(G) is as before. Definition 3 looks for an instance mapping μ with μ(H) ⊆ cl_R(G); μ(_:y) = tweety works.

**Semantics' side.** **Definition 11** interprets a blank-node graph through its ground instances over `N`: ⟦H⟧ = ⟨∇{⟦μ(H)⟧⁺ : μ}, ⊔{⟦μ(H)⟧⁻ : μ}⟩, μ ranging over the mappings bnodes(H) → N. There are seven mappings, one per name in `N`. Each instance μ(H) is a single triple, so each ⟦μ(H)⟧⁻ is generated by the single pair ⟨∅, {μ(H)}⟩, and the adjunction over the seven of them is generated by one pair whose succedent holds all seven instances. That pair is what "something is a Flier" denies: it is in 𝕀_R exactly when cl_R(Γ) contains at least one of the seven, i.e. when Γ makes *some* name in `N` a Flier. This is Hlobil's ∃-clause with instances over `N` in place of objects.

In [4]:
H_b = frozenset({('_y', 'type', 'Flier')})
assert U.admissible(G_a, H_b)
print("H =", graph(H_b))

print("\n-- Definition 11: the instance mappings and instances of H --")
for mu, h in zip(mappings(H_b, U), instances(H_b, U)):
    print(f"  μ = {mapping(mu):22s} μ(H) = {graph(h)}")
negH = content_neg(H_b, U)
print(f"\n[[H]]- = ⊔ over the {len(mappings(H_b, U))} mappings, generated by {len(negH)} pair(s):")
print(pairs(negH))

print("\n-- practice's side --")
cl = closure(G_a)
print("cl_R(G) \\ G =", graph(cl - G_a))
print("some μ with μ(H) ⊆ cl_R(G):", simply_entails(cl - {BOT}, H_b), " (μ(_:y) = tweety)")
rules_b = r_entails(closure, G_a, H_b)

print("\n-- semantics' side --")
F = adj(content_pos(G_a, U), negH)
print(f"[[G]]+ ⊔ [[H]]-: {len(F)} pair(s):")
sem_b = report(closure, F)
assert sem_b == iss_entails(good, G_a, H_b, U)

print(); print(verdict_line("B: blank node in H", sem_b, rules_b))
verdicts.append(("B: blank node in H", sem_b, rules_b))

H = {(_:y, type, Flier)}

-- Definition 11: the instance mappings and instances of H --
  μ = {_:y ↦ tweety}         μ(H) = {(tweety, type, Flier)}
  μ = {_:y ↦ Bird}           μ(H) = {(Bird, type, Flier)}
  μ = {_:y ↦ Flier}          μ(H) = {(Flier, type, Flier)}
  μ = {_:y ↦ type}           μ(H) = {(type, type, Flier)}
  μ = {_:y ↦ subClassOf}     μ(H) = {(subClassOf, type, Flier)}
  μ = {_:y ↦ disjointWith}   μ(H) = {(disjointWith, type, Flier)}
  μ = {_:y ↦ s}              μ(H) = {(s, type, Flier)}

[[H]]- = ⊔ over the 7 mappings, generated by 1 pair(s):
⟨∅, {(Bird, type, Flier), (Flier, type, Flier), (disjointWith, type, Flier), (s, type, Flier), (subClassOf, type, Flier), (tweety, type, Flier), (type, type, Flier)}⟩

-- practice's side --
cl_R(G) \ G = {(tweety, type, Flier)}
some μ with μ(H) ⊆ cl_R(G): True  (μ(_:y) = tweety)

-- semantics' side --
[[G]]+ ⊔ [[H]]-: 1 pair(s):
  ✓ ⟨{(Bird, subClassOf, Flier), (tweety, type, Bird)}, {(Bird, type, Flier), (Flier, type, Flier), (dis

## Case C: a blank node in `G`

`G` = {(_:x, type, Bird), (Bird, subClassOf, Flier)}: "something is a Bird, and Birds are Fliers".

**Semantics' side, the positive content.** By Definition 11, ⟦G⟧⁺ = ∇{⟦ν(G)⟧⁺ : ν}, over the seven instance mappings ν : {_:x} → N. Each ⟦ν(G)⟧⁺ is generated by ⟨ν(G), ∅⟩; the power-symjunction over seven of them is generated by the unions ⟨⋃_{ν∈x} ν(G), ∅⟩ for every non-empty set x of mappings: 2⁷ − 1 = 127 pairs (Lemma 3 describes exactly this shape; the notebook enumerates it from Definition 6 without using the lemma). The seven singletons are printed in full; the other 120 are described by their x.

Then two different `H`:

* **C1**, `H` = {(tweety, type, Flier)}. The rules do not derive it: cl_R(G) contains (_:x, type, Flier), not (tweety, type, Flier), and no instance mapping helps because `H` is ground. The semantics agrees, and the pair that shows it is the one the *only-if* direction of Lemma 7 constructs: take the **Skolem instance** ν_sk(_:x) = s (Lemma 5: `s` is an IRI of `N` occurring nowhere in `G`, `H` or `V`, which is what Definition 12 guarantees) and the singleton pair ⟨ν_sk(G), {(tweety, type, Flier)}⟩. Its closure makes `s` a Flier, not tweety; the pair is not in 𝕀_R; so ⟦G⟧ ⊭ ⟦H⟧. Other instances fail too (ν(_:x) = Bird, say), but ν_sk is the one that *must* fail whenever the rules do not derive `H`, because `s` is fresh (Remark 9: Skolemization is sound in the antecedent).

* **C2**, `H` = {(_:y, type, Flier)}. Now every instance ν(G) makes ν(_:x) a Flier, and ν(_:x) is one of the seven names that ⟦H⟧⁻'s single pair denies; so all 127 pairs are in 𝕀_R. On the practice's side, μ(_:y) = _:x witnesses simple entailment in cl_R(G).

In [5]:
G_c = frozenset({('_x', 'type', 'Bird'), ('Bird', 'subClassOf', 'Flier')})
assert U.admissible(G_c, H_a) and U.admissible(G_c, H_b)
print("G =", graph(G_c))

print("\n-- Definition 11: instance mappings ν and instances ν(G) --")
labelled = []
for nu, g in zip(mappings(G_c, U), instances(G_c, U)):
    tag = "  ← ν_sk, the Skolem instance (Lemma 5)" if nu['_x'] == 's' else ""
    print(f"  ν = {mapping(nu):24s} ν(G) = {graph(g)}{tag}")
    labelled.append((nu['_x'], g))
posG = content_pos(G_c, U)
print(f"\n[[G]]+ = ∇ over the 7 instances, generated by {len(posG)} pairs (2^7 - 1).")
print("The 7 singleton pairs <ν(G), ∅>:")
singles = [p for p in sorted(posG, key=show.pair_key) if len(union_labels(p[0], labelled)) == 1]
print(pairs(singles))
print("\nAll 127 pairs <⋃_{ν∈x} ν(G), ∅>, each described by x as the set of ν(_:x) values, grouped by |x|:")
by_size = {}
for p in sorted(posG, key=show.pair_key):
    x = union_labels(p[0], labelled)
    by_size.setdefault(len(x), []).append('{' + ','.join(x) + '}')
for k in sorted(by_size):
    print(f"  |x| = {k} ({len(by_size[k])} pairs, |Γ| = {k + 1}):  " + ' '.join(by_size[k]))

G = {(Bird, subClassOf, Flier), (_:x, type, Bird)}

-- Definition 11: instance mappings ν and instances ν(G) --
  ν = {_:x ↦ tweety}           ν(G) = {(Bird, subClassOf, Flier), (tweety, type, Bird)}
  ν = {_:x ↦ Bird}             ν(G) = {(Bird, subClassOf, Flier), (Bird, type, Bird)}
  ν = {_:x ↦ Flier}            ν(G) = {(Bird, subClassOf, Flier), (Flier, type, Bird)}
  ν = {_:x ↦ type}             ν(G) = {(Bird, subClassOf, Flier), (type, type, Bird)}
  ν = {_:x ↦ subClassOf}       ν(G) = {(Bird, subClassOf, Flier), (subClassOf, type, Bird)}
  ν = {_:x ↦ disjointWith}     ν(G) = {(Bird, subClassOf, Flier), (disjointWith, type, Bird)}
  ν = {_:x ↦ s}                ν(G) = {(Bird, subClassOf, Flier), (s, type, Bird)}  ← ν_sk, the Skolem instance (Lemma 5)

[[G]]+ = ∇ over the 7 instances, generated by 127 pairs (2^7 - 1).
The 7 singleton pairs <ν(G), ∅>:
⟨{(Bird, subClassOf, Flier), (Bird, type, Bird)}, ∅⟩
⟨{(Bird, subClassOf, Flier), (Flier, type, Bird)}, ∅⟩
⟨{(Bird, subClassOf, Flie

In [6]:
print("== C1:  H =", graph(H_a), "==")
print("\n-- practice's side --")
cl = closure(G_c)
print("cl_R(G) \\ G =", graph(cl - G_c))
print("some μ with μ(H) ⊆ cl_R(G):", simply_entails(cl - {BOT}, H_a), " (H is ground; (tweety, type, Flier) is not there)")
rules_c1 = r_entails(closure, G_c, H_a)

print("\n-- semantics' side --")
negH = content_neg(H_a, U)
F = adj(posG, negH)
n_bad = sum(1 for p in F if not good(p))
print(f"[[G]]+ ⊔ [[H]]-: {len(F)} pairs, of which {n_bad} are not in I_R.")
print("The Skolem pair, x = {s} (the pair Lemma 7's only-if direction constructs):")
sk = [p for p in F if union_labels(p[0], labelled) == ['s']]
sem_c1_sk = report(closure, sk)
print("The one singleton pair that IS in I_R, x = {tweety}:")
report(closure, [p for p in F if union_labels(p[0], labelled) == ['tweety']])
with_t = [p for p in F if 'tweety' in union_labels(p[0], labelled)]
without = [p for p in F if 'tweety' not in union_labels(p[0], labelled)]
print(f"Pairs whose x contains tweety: {len(with_t)}, all in I_R: {all(good(p) for p in with_t)} "
      f"(Γ contains ν(G) for ν(_:x) = tweety, so cl_R(Γ) has (tweety, type, Flier)).")
print(f"Pairs whose x does not:        {len(without)}, none in I_R: {not any(good(p) for p in without)}.")
sem_c1 = iss_entails(good, G_c, H_a, U)
assert sem_c1 == (n_bad == 0)

print(); print(verdict_line("C1: blank node in G, ground H", sem_c1, rules_c1))
verdicts.append(("C1: blank node in G, ground H", sem_c1, rules_c1))

== C1:  H = {(tweety, type, Flier)} ==

-- practice's side --
cl_R(G) \ G = {(_:x, type, Flier)}
some μ with μ(H) ⊆ cl_R(G): False  (H is ground; (tweety, type, Flier) is not there)

-- semantics' side --
[[G]]+ ⊔ [[H]]-: 127 pairs, of which 63 are not in I_R.
The Skolem pair, x = {s} (the pair Lemma 7's only-if direction constructs):
  ✗ ⟨{(Bird, subClassOf, Flier), (s, type, Bird)}, {(tweety, type, Flier)}⟩
      Γ ∩ Δ = ∅, ⊥ ∉ cl_R(Γ), and Δ ∩ cl_R(Γ) = ∅; cl_R(Γ) ∖ Γ = {(s, type, Flier)}
The one singleton pair that IS in I_R, x = {tweety}:
  ✓ ⟨{(Bird, subClassOf, Flier), (tweety, type, Bird)}, {(tweety, type, Flier)}⟩
      (tweety, type, Flier) ∈ Δ ∩ cl_R(Γ)  (Definition 13)
Pairs whose x contains tweety: 64, all in I_R: True (Γ contains ν(G) for ν(_:x) = tweety, so cl_R(Γ) has (tweety, type, Flier)).
Pairs whose x does not:        63, none in I_R: True.

C1: blank node in G, ground H            semantics False  rules False  agree


In [7]:
print("== C2:  H =", graph(H_b), "==")
print("\n-- practice's side --")
cl = closure(G_c)
print("some μ with μ(H) ⊆ cl_R(G):", simply_entails(cl - {BOT}, H_b), " (μ(_:y) = _:x)")
rules_c2 = r_entails(closure, G_c, H_b)

print("\n-- semantics' side --")
negH = content_neg(H_b, U)
F = adj(posG, negH)
print(f"[[G]]+ ⊔ [[H]]-: {len(F)} pairs. The 7 singleton pairs, with reasons:")
report(closure, [p for p in F if len(union_labels(p[0], labelled)) == 1])
sem_c2 = iss_entails(good, G_c, H_b, U)
print(f"All {len(F)} pairs in I_R: {sem_c2}  (every union contains some ν(G), whose closure makes ν(_:x) a Flier)")

print(); print(verdict_line("C2: blank node in G, blank node in H", sem_c2, rules_c2))
verdicts.append(("C2: blank node in G, blank node in H", sem_c2, rules_c2))

== C2:  H = {(_:y, type, Flier)} ==

-- practice's side --
some μ with μ(H) ⊆ cl_R(G): True  (μ(_:y) = _:x)

-- semantics' side --
[[G]]+ ⊔ [[H]]-: 127 pairs. The 7 singleton pairs, with reasons:
  ✓ ⟨{(Bird, subClassOf, Flier), (Bird, type, Bird)}, {(Bird, type, Flier), (Flier, type, Flier), (disjointWith, type, Flier), (s, type, Flier), (subClassOf, type, Flier), (tweety, type, Flier), (type, type, Flier)}⟩
      (Bird, type, Flier) ∈ Δ ∩ cl_R(Γ)  (Definition 13)
  ✓ ⟨{(Bird, subClassOf, Flier), (Flier, type, Bird)}, {(Bird, type, Flier), (Flier, type, Flier), (disjointWith, type, Flier), (s, type, Flier), (subClassOf, type, Flier), (tweety, type, Flier), (type, type, Flier)}⟩
      (Flier, type, Flier) ∈ Δ ∩ cl_R(Γ)  (Definition 13)
  ✓ ⟨{(Bird, subClassOf, Flier), (disjointWith, type, Bird)}, {(Bird, type, Flier), (Flier, type, Flier), (disjointWith, type, Flier), (s, type, Flier), (subClassOf, type, Flier), (tweety, type, Flier), (type, type, Flier)}⟩
      (disjointWith, type, Fl

## Case D: an incoherent `G` (Proposition 2)

Add (Bird, disjointWith, Flier) to the ground `G` of case A. Now cl_R(G) contains (tweety, type, Flier) by rdfs9 and then ⊥ by cax-dw: `G` is R-inconsistent (Definition 4).

**Proposition 2** says that the semantics' verdict on `G` alone, ⟦G⟧ ⊨ ∅, coincides with R-inconsistency. Content entailment with the empty set of succedent contents (Definition 7) adjoins ⟦G⟧⁺ to the unit role, generated by ⟨∅, ∅⟩; so the pairs to test are the generating pairs of ⟦G⟧⁺ themselves, here the single ⟨G, ∅⟩. It is in 𝕀_R because ⊥ ∈ cl_R(G) (Definition 13: an R-inconsistent Γ implies everything, the empty succedent included). For comparison, the consistent `G` of case A is not incoherent.

Explosion is a property of the regime's format, not of the semantics (the paragraph after Definition 13): the incoherent `G` entails every `H`, on both sides, including one unrelated to it.

In [8]:
G_d = G_a | {('Bird', 'disjointWith', 'Flier')}
assert U.admissible(G_d)
print("G =", graph(G_d))

print("\n-- practice's side --")
cl = closure(G_d)
print("cl_R(G) \\ G =", graph(cl - G_d))
rules_d = r_inconsistent(closure, G_d)
print("G is R-inconsistent:", rules_d)

print("\n-- semantics' side: [[G]] |~ ∅ --")
posG = content_pos(G_d, U)
F = adj(posG, UNIT)
print(f"[[G]]+ ⊔ (unit): {len(F)} pair(s):")
sem_d = report(closure, F)
assert sem_d == iss_incoherent(good, G_d, U)
print(); print(verdict_line("D: incoherent G (Prop. 2)", sem_d, rules_d))
verdicts.append(("D: incoherent G (Prop. 2)", sem_d, rules_d))

print("\n-- for comparison, the consistent G of case A --")
print("[[G_A]] |~ ∅:", iss_incoherent(good, G_a, U), "   G_A R-inconsistent:", r_inconsistent(closure, G_a))
verdicts.append(("D': consistent G of case A, [[G]] |~ ∅", iss_incoherent(good, G_a, U), r_inconsistent(closure, G_a)))

print("\n-- explosion: the incoherent G entails an unrelated H --")
H_x = frozenset({('Flier', 'type', 'tweety')})
print("H =", graph(H_x))
sem_dx = iss_entails(good, G_d, H_x, U); rules_dx = r_entails(closure, G_d, H_x)
print(verdict_line("D'': incoherent G, unrelated H", sem_dx, rules_dx))
verdicts.append(("D'': incoherent G, unrelated H", sem_dx, rules_dx))

G = {(Bird, disjointWith, Flier), (Bird, subClassOf, Flier), (tweety, type, Bird)}

-- practice's side --
cl_R(G) \ G = {(tweety, type, Flier), ⊥}
G is R-inconsistent: True

-- semantics' side: [[G]] |~ ∅ --
[[G]]+ ⊔ (unit): 1 pair(s):
  ✓ ⟨{(Bird, disjointWith, Flier), (Bird, subClassOf, Flier), (tweety, type, Bird)}, ∅⟩
      ⊥ ∈ cl_R(Γ): Γ is R-inconsistent  (Definition 13)

D: incoherent G (Prop. 2)                semantics True   rules True   agree

-- for comparison, the consistent G of case A --
[[G_A]] |~ ∅: False    G_A R-inconsistent: False

-- explosion: the incoherent G entails an unrelated H --
H = {(Flier, type, tweety)}
D'': incoherent G, unrelated H           semantics True   rules True   agree


## Summary

Every case gives the same verdict on both sides, as Theorem 2 and Proposition 2 say it must. The last cell writes the verdicts to `results/walkthrough.txt`, the repository's ledger of runs.

In [9]:
lines = [verdict_line(label, s, r) for label, s, r in verdicts]
print('\n'.join(lines))
assert all(s == r for _, s, r in verdicts)
show.write_log('../results/walkthrough.txt',
               [f"walkthrough: universe N = {U.N}, regime rdfs9 + cax-dw"] + lines +
               [f"summary: {len(verdicts)} cases, {sum(s != r for _, s, r in verdicts)} disagreements"])

A: ground G, ground H                    semantics True   rules True   agree
B: blank node in H                       semantics True   rules True   agree
C1: blank node in G, ground H            semantics False  rules False  agree
C2: blank node in G, blank node in H     semantics True   rules True   agree
D: incoherent G (Prop. 2)                semantics True   rules True   agree
D': consistent G of case A, [[G]] |~ ∅   semantics False  rules False  agree
D'': incoherent G, unrelated H           semantics True   rules True   agree
wrote ../results/walkthrough.txt (9 lines)
